# LeetCode 2190 — Most Frequent Number Following Key In an Array bằng Python

Link đề: https://leetcode.com/problems/most-frequent-number-following-key-in-an-array/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Counting

Skeleton LeetCode:

```python
class Solution:
    def mostFrequent(self, nums: List[int], key: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm số `target` **đứng ngay sau** `key` nhiều lần nhất (đếm theo cặp liền kề `(nums[i], nums[i+1])`).
2. Biết dùng `dict`, `collections.Counter` để **đếm tần suất** và lấy phần tử có tần suất cao nhất.
3. Viết được 4 cách: brute force (duyệt từng `target`), đếm bằng `Counter`, đếm bằng mảng tần suất, và `zip` + `most_common`.
4. Nắm vững lỗi kinh điển: **đi quá biên** `nums[i + 1]` khi `i` là phần tử cuối, và nhầm "đếm `key`" với "đếm số đứng sau `key`".
5. Tự trace, tự test ngẫu nhiên đối chiếu nhiều cách và làm 5 bài tập vận dụng.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` (đánh chỉ số từ 0) và một số `key` chắc chắn có trong `nums`.

Với mỗi số nguyên `target` xuất hiện trong `nums`, ta đếm xem có bao nhiêu chỉ số `i` (với `0 <= i <= n - 2`) thỏa **đồng thời**:

- `nums[i] == key`, và
- `nums[i + 1] == target`.

Hãy trả về `target` có **số lần đếm lớn nhất**. Đề bảo đảm đáp án là **duy nhất** (không có hòa).

| Ví dụ | nums | key | Output | Giải thích |
|---|---|---|---|---|
| 1 | `[1,100,200,1,100]` | `1` | `100` | Sau `1` là `100` (2 lần) và `200` (1 lần). |
| 2 | `[2,2,2,2,3]` | `2` | `2` | Sau `2` là `2` (3 lần), `3` (1 lần). |

Constraints:

- `2 <= nums.length <= 1000`
- `1 <= nums[i] <= 1000`
- Đề bảo đảm đáp án là duy nhất.
- `key` là một số nguyên có trong `nums`.

Chú ý ở ví dụ 2: số `2` vừa là `key`, vừa là `target`. Điều này hoàn toàn hợp lệ.

In [1]:
examples = [([1, 100, 200, 1, 100], 1, 100), ([2, 2, 2, 2, 3], 2, 2)]
for nums, key, expected in examples:
    after = [nums[i + 1] for i in range(len(nums) - 1) if nums[i] == key]
    print("nums =", nums, "| key =", key, "| mong đợi =", expected, "| các số đứng sau key:", after)

nums = [1, 100, 200, 1, 100] | key = 1 | mong đợi = 100 | các số đứng sau key: [100, 100]
nums = [2, 2, 2, 2, 3] | key = 2 | mong đợi = 2 | các số đứng sau key: [2, 2, 2, 3]


## 3. Kiến thức nền

Bài này cần ba công cụ Python: **đếm bằng dict**, **`Counter`**, và **duyệt cặp liền kề** (`zip`, hoặc chỉ số `i` và `i + 1`).

### 3.1. `dict` đếm tần suất

Một `dict` lưu cặp `khóa → giá trị`. Để đếm, ta dùng khóa là phần tử, giá trị là số lần đã thấy. `d.get(x, 0)` trả về `0` nếu `x` chưa có khóa trong `d`, nhờ vậy không bị `KeyError`.

In [2]:
d = {}
for x in [5, 7, 5, 5, 7, 9]:
    d[x] = d.get(x, 0) + 1
print(d)
print(d.get(5, 0), d.get(100, 0))   # khóa không có -> 0
print(max(d, key=d.get))            # khóa có giá trị lớn nhất

{5: 3, 7: 2, 9: 1}
3 0
5


### 3.2. `collections.Counter`

`Counter` là dict chuyên đếm: `Counter(iterable)` đếm hết một lượt. `c[x]` của khóa chưa có trả về `0` (không lỗi). `c.most_common(1)` trả về danh sách `[(phần_tử, số_lần)]` cho phần tử phổ biến nhất.

In [3]:
from collections import Counter

c = Counter([5, 7, 5, 5, 7, 9])
print(c)
print(c[5], c[100])
print(c.most_common(1))      # [(5, 3)]
print(c.most_common(2))

Counter({5: 3, 7: 2, 9: 1})
3 0
[(5, 3)]
[(5, 3), (7, 2)]


### 3.3. Duyệt cặp liền kề: chỉ số `i` và `i + 1`

Để so sánh `nums[i]` với `nums[i + 1]`, `i` chỉ được chạy đến `n - 2` (nếu không, `nums[i + 1]` vượt biên). Do đó dùng `range(len(nums) - 1)`. Một cách khác gọn hơn là `zip(nums, nums[1:])`, nó tự dừng khi dãy ngắn hơn hết.

In [4]:
nums = [1, 100, 200, 1, 100]
for i in range(len(nums) - 1):
    print("i =", i, "->", (nums[i], nums[i + 1]))
print(list(zip(nums, nums[1:])))

i = 0 -> (1, 100)
i = 1 -> (100, 200)
i = 2 -> (200, 1)
i = 3 -> (1, 100)
[(1, 100), (100, 200), (200, 1), (1, 100)]


### 3.4. List comprehension có điều kiện

`[b for a, b in pairs if a == key]` nghĩa là: duyệt từng cặp `(a, b)`, giữ `b` khi `a == key`. Đây chính là "lọc ra các số đứng ngay sau `key`".

In [5]:
nums, key = [2, 2, 2, 2, 3], 2
pairs = list(zip(nums, nums[1:]))
print(pairs)
print([b for a, b in pairs if a == key])

[(2, 2), (2, 2), (2, 2), (2, 3)]
[2, 2, 2, 3]


## 4. Ý tưởng nền tảng — tính bằng tay

Lấy `nums = [1,100,200,1,100]`, `key = 1`.

1. Tìm mọi vị trí của `key = 1`: chỉ số `0` và `3`.
2. Số đứng ngay sau mỗi vị trí đó: `nums[1] = 100`, `nums[4] = 100`.
3. Đếm: `100` xuất hiện `2` lần.
4. Vậy đáp án là `100`.

Tóm lại, bài toán thực chất là: **lấy ra danh sách các số đứng ngay sau `key`, rồi tìm phần tử xuất hiện nhiều nhất trong danh sách đó** (bài "tìm mode").

Lưu ý: nếu `key` nằm ở **cuối** mảng thì nó không có số đứng sau, nên vị trí đó không đóng góp gì.

In [6]:
nums, key = [1, 100, 200, 1, 100], 1
positions = [i for i, v in enumerate(nums) if v == key]
followers = [nums[i + 1] for i in positions if i + 1 < len(nums)]
print("vị trí của key:", positions)
print("số đứng sau key:", followers)
print("phổ biến nhất:", max(set(followers), key=followers.count))

vị trí của key: [0, 3]
số đứng sau key: [100, 100]
phổ biến nhất: 100


## 5. Cách 1 — Brute Force: thử từng `target`

Ý tưởng thẳng nhất theo đúng đề: với **mỗi** giá trị `target` khác nhau trong `nums`, duyệt lại cả mảng và đếm số `i` thỏa `nums[i] == key` và `nums[i+1] == target`. Giữ `target` có đếm lớn nhất.

In [7]:
class SolutionBrute:
    def mostFrequent(self, nums, key):
        best_target, best_count = None, -1
        for target in set(nums):                 # mỗi giá trị khác nhau
            count = 0
            for i in range(len(nums) - 1):       # duyệt lại cả mảng
                if nums[i] == key and nums[i + 1] == target:
                    count += 1
            if count > best_count:
                best_target, best_count = target, count
        return best_target

sol = SolutionBrute()
print(sol.mostFrequent([1, 100, 200, 1, 100], 1))
print(sol.mostFrequent([2, 2, 2, 2, 3], 2))

100
2


**Độ phức tạp:** với `u` là số giá trị khác nhau, thời gian `O(n · u)`, tệ nhất `O(n²)` (khi mọi phần tử khác nhau). Bộ nhớ `O(u)` cho `set(nums)`.

**Nút thắt:** ta duyệt lại toàn bộ mảng cho **mỗi** `target`, trong khi thông tin cần thiết (cặp liền kề) chỉ cần nhìn **một lần**. Với `n = 1000` vẫn chạy nhanh, nhưng nếu `n = 10^6` thì không thể chấp nhận.

## 6. Ý tưởng thuật toán — bước đột phá

**Insight:** mỗi chỉ số `i` chỉ cho ta **một** cặp `(nums[i], nums[i+1])`. Nếu `nums[i] == key` thì cặp đó "bỏ một phiếu" cho `nums[i+1]`. Vậy ta chỉ cần **một lượt duyệt**, bỏ phiếu vào bảng đếm, rồi chọn người nhiều phiếu nhất.

Quy tắc "nếu… thì…":

- **Nếu** `nums[i] == key` **thì** `count[nums[i+1]] += 1`.
- **Nếu** `nums[i] != key` **thì** bỏ qua.
- Sau khi duyệt xong, đáp án là khóa có `count` lớn nhất.

**Invariant:** sau khi xử lý các chỉ số `0..i`, `count[t]` đúng bằng số lần `t` đứng sau `key` trong các cặp `(0,1), (1,2), …, (i,i+1)`.

## 7. Các cách tiếp theo

Ba cách dưới đây đều **một lượt duyệt**, chỉ khác công cụ đếm: `dict/Counter` thủ công, mảng đếm theo giá trị, và `zip + most_common`.

### Cách 2 — Dict đếm thủ công (một lượt)

- Tạo `count = {}`.
- Với `i` từ `0` đến `n - 2`: nếu `nums[i] == key` thì `count[nums[i+1]] += 1`.
- Cuối cùng chọn khóa có `count` lớn nhất bằng `max(count, key=count.get)`.

In [8]:
class SolutionHash:
    def mostFrequent(self, nums, key):
        count = {}
        for i in range(len(nums) - 1):
            if nums[i] == key:
                nxt = nums[i + 1]
                count[nxt] = count.get(nxt, 0) + 1
        return max(count, key=count.get)

sol = SolutionHash()
print(sol.mostFrequent([1, 100, 200, 1, 100], 1))
print(sol.mostFrequent([2, 2, 2, 2, 3], 2))

100
2


**Độ phức tạp:** Time `O(n)`, Space `O(u)` (số giá trị khác nhau đứng sau `key`).

### Cách 3 — Mảng đếm theo giá trị (vì `1 <= nums[i] <= 1000`)

Constraints nói giá trị nằm trong `[1, 1000]`, nên có thể thay dict bằng **mảng** `cnt` kích thước `1001`, truy cập `cnt[v]` trực tiếp. Vừa nhanh vừa không cần `import`. Trong lúc đếm, ta cập nhật luôn "người dẫn đầu" nên không cần quét lại bảng đếm.

In [9]:
class SolutionArray:
    def mostFrequent(self, nums, key):
        cnt = [0] * 1001
        best, best_count = -1, 0
        for i in range(len(nums) - 1):
            if nums[i] == key:
                v = nums[i + 1]
                cnt[v] += 1
                if cnt[v] > best_count:     # cập nhật người dẫn đầu ngay khi đếm
                    best, best_count = v, cnt[v]
        return best

sol = SolutionArray()
print(sol.mostFrequent([1, 100, 200, 1, 100], 1))
print(sol.mostFrequent([2, 2, 2, 2, 3], 2))

100
2


**Độ phức tạp:** Time `O(n)`, Space `O(V)` với `V = 1001` là miền giá trị (hằng số theo constraints).

Lưu ý: vì đề đảm bảo đáp án duy nhất, "người dẫn đầu" cuối cùng chính là đáp án. Nếu có hòa, kết quả phụ thuộc thứ tự cập nhật (người đạt mức cao trước giữ vị trí).

### Cách 4 — `zip` + `Counter.most_common` (gọn, kiểu Pythonic)

`zip(nums, nums[1:])` sinh các cặp liền kề. Ta lọc cặp có phần tử đầu bằng `key`, đếm phần tử thứ hai bằng `Counter`, rồi lấy `most_common(1)`.

In [10]:
from collections import Counter

class SolutionZip:
    def mostFrequent(self, nums, key):
        c = Counter(b for a, b in zip(nums, nums[1:]) if a == key)
        return c.most_common(1)[0][0]

sol = SolutionZip()
print(sol.mostFrequent([1, 100, 200, 1, 100], 1))
print(sol.mostFrequent([2, 2, 2, 2, 3], 2))

100
2


**Độ phức tạp:** Time `O(n)`, Space `O(n)` vì `nums[1:]` tạo bản sao (có thể dùng `itertools.islice` để tránh). Đây là cách ngắn nhất, rất hợp khi thi đấu hoặc phỏng vấn nhanh.

## 8. TRACE từng bước

Trace cách đếm một lượt với `nums = [2,2,2,2,3]`, `key = 2`:

| i | nums[i] | nums[i+1] | nums[i]==key? | count sau bước |
|---|---|---|---|---|
| 0 | 2 | 2 | có | `{2:1}` |
| 1 | 2 | 2 | có | `{2:2}` |
| 2 | 2 | 2 | có | `{2:3}` |
| 3 | 2 | 3 | có | `{2:3, 3:1}` |

Kết quả: `2` (3 phiếu) thắng `3` (1 phiếu). Hàm dưới đây in đúng các bước đó.

In [11]:
def trace_count(nums, key):
    count = {}
    for i in range(len(nums) - 1):
        a, b = nums[i], nums[i + 1]
        hit = (a == key)
        if hit:
            count[b] = count.get(b, 0) + 1
        print(f"i={i} | nums[i]={a} nums[i+1]={b} | khớp key? {hit} | count={count}")
    ans = max(count, key=count.get)
    print("=> đáp án:", ans)
    return ans

trace_count([2, 2, 2, 2, 3], 2)

i=0 | nums[i]=2 nums[i+1]=2 | khớp key? True | count={2: 1}
i=1 | nums[i]=2 nums[i+1]=2 | khớp key? True | count={2: 2}
i=2 | nums[i]=2 nums[i+1]=2 | khớp key? True | count={2: 3}
i=3 | nums[i]=2 nums[i+1]=3 | khớp key? True | count={2: 3, 3: 1}
=> đáp án: 2


2

In [12]:
trace_count([1, 100, 200, 1, 100], 1)

i=0 | nums[i]=1 nums[i+1]=100 | khớp key? True | count={100: 1}
i=1 | nums[i]=100 nums[i+1]=200 | khớp key? False | count={100: 1}
i=2 | nums[i]=200 nums[i+1]=1 | khớp key? False | count={100: 1}
i=3 | nums[i]=1 nums[i+1]=100 | khớp key? True | count={100: 2}
=> đáp án: 100


100

## 9. Code nộp LeetCode

Dùng cách 2 (dict thủ công) — rõ ràng, không phụ thuộc thư viện, và độ phức tạp tối ưu.

In [13]:
from typing import List

class Solution:
    def mostFrequent(self, nums: List[int], key: int) -> int:
        count = {}
        for i in range(len(nums) - 1):
            if nums[i] == key:
                nxt = nums[i + 1]
                count[nxt] = count.get(nxt, 0) + 1
        return max(count, key=count.get)

print(Solution().mostFrequent([1, 100, 200, 1, 100], 1))
print(Solution().mostFrequent([2, 2, 2, 2, 3], 2))

100
2


## 10. Vì sao thuật toán đúng?

- **Phủ đủ mọi cặp:** vòng `for i in range(n - 1)` đi qua mọi cặp liền kề `(i, i+1)` đúng một lần, đúng như định nghĩa trong đề (`0 <= i <= n-2`).
- **Chỉ đếm cặp hợp lệ:** ta chỉ cộng phiếu khi `nums[i] == key`; do đó `count[t]` đúng bằng số chỉ số `i` với `nums[i] == key` và `nums[i+1] == t`.
- **Các `target` chưa từng đứng sau `key`** có đếm `0`, không thể thắng một `target` có đếm `>= 1` — mà `key` chắc chắn xuất hiện, nên miễn là có ít nhất một cặp thì `count` không rỗng. (Đề bảo đảm có đáp án, nên `key` không bị kẹt chỉ ở cuối mảng.)
- **Chọn max:** đề bảo đảm đáp án duy nhất, nên `max(count, key=count.get)` trả đúng người duy nhất đó.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)`, `u` là số giá trị khác nhau đứng sau `key`, `V = 1000` là miền giá trị.

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(n · u)` (tệ nhất `O(n²)`) | `O(u)` |
| 2. Dict đếm | `O(n)` | `O(u)` |
| 3. Mảng đếm | `O(n)` | `O(V)` = `O(1)` hằng số |
| 4. zip + Counter | `O(n)` | `O(n)` |

Với `n <= 1000` mọi cách đều qua, nhưng cách một lượt là thói quen tốt cần rèn: **đếm trong một lần duyệt**.

## 12. Bộ test

Kiểm tra ví dụ đề, edge case, rồi đối chiếu cả 4 cách trên input ngẫu nhiên có `seed`.

In [14]:
solutions = [SolutionBrute(), SolutionHash(), SolutionArray(), SolutionZip(), Solution()]

# ví dụ đề
for s in solutions:
    assert s.mostFrequent([1, 100, 200, 1, 100], 1) == 100
    assert s.mostFrequent([2, 2, 2, 2, 3], 2) == 2
print("ví dụ đề: OK")

ví dụ đề: OK


In [15]:
# edge case
for s in solutions:
    assert s.mostFrequent([1, 2], 1) == 2                 # chỉ có một cặp
    assert s.mostFrequent([5, 5], 5) == 5                 # key và target trùng nhau
    assert s.mostFrequent([3, 7, 3, 7, 3, 9, 3], 3) == 7  # key ở cuối mảng: không có số đứng sau
    assert s.mostFrequent([4, 1, 1, 1, 2], 1) == 1        # sau key=1: 1,1,2 -> 1 thắng
    assert s.mostFrequent([9, 9, 9, 9], 9) == 9
print("edge case: OK")

edge case: OK


In [16]:
import random
random.seed(2190)

def unique_best(nums, key):
    """Trả về True nếu đáp án duy nhất (để khớp đảm bảo của đề)."""
    c = Counter(nums[i + 1] for i in range(len(nums) - 1) if nums[i] == key)
    if not c:
        return False
    top = c.most_common(2)
    return len(top) == 1 or top[0][1] > top[1][1]

tested = 0
while tested < 600:
    n = random.randint(2, 30)
    nums = [random.randint(1, 6) for _ in range(n)]
    key = random.choice(nums)
    if not unique_best(nums, key):
        continue
    results = {type(s).__name__: s.mostFrequent(nums, key) for s in solutions}
    assert len(set(results.values())) == 1, (nums, key, results)
    tested += 1
print("random đối chiếu 5 cách:", tested, "ca, tất cả khớp")

random đối chiếu 5 cách: 600 ca, tất cả khớp


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Vượt biên khi truy cập `nums[i + 1]`

Nếu để `i` chạy tới `n - 1`, thì `nums[i + 1]` là chỉ số `n` → `IndexError`.

In [17]:
def wrong_out_of_range(nums, key):
    count = {}
    for i in range(len(nums)):            # SAI: phải là len(nums) - 1
        if nums[i] == key:
            count[nums[i + 1]] = count.get(nums[i + 1], 0) + 1
    return max(count, key=count.get)

try:
    print(wrong_out_of_range([3, 7, 3], 3))
except IndexError as e:
    print("IndexError:", e)
print("đúng:", Solution().mostFrequent([3, 7, 3], 3))

IndexError: list index out of range
đúng: 7


### Lỗi 2 — Đếm số lần `key` xuất hiện thay vì đếm số đứng sau `key`

Người mới đọc lướt và dùng `Counter(nums)` rồi lấy phổ biến nhất. Kết quả sai vì ta cần phần tử **đứng sau** `key`, không phải phần tử xuất hiện nhiều nhất toàn mảng.

In [18]:
def wrong_count_all(nums, key):
    return Counter(nums).most_common(1)[0][0]

nums, key = [1, 100, 200, 1, 100, 7, 7, 7], 1
print("sai :", wrong_count_all(nums, key))      # 7 (nhiều nhất toàn mảng)
print("đúng:", Solution().mostFrequent(nums, key))  # 100

sai : 7
đúng: 100


### Lỗi 3 — Đếm cả những chỗ không đứng sau `key`

Quên điều kiện `nums[i] == key` thì ta đếm mọi cặp liền kề, tức là tìm "số hay đứng sau một số bất kỳ".

In [19]:
def wrong_no_filter(nums, key):
    c = Counter(nums[i + 1] for i in range(len(nums) - 1))   # thiếu if nums[i] == key
    return c.most_common(1)[0][0]

nums, key = [1, 100, 5, 8, 5, 8, 5, 8], 1
print("sai :", wrong_no_filter(nums, key))        # 8 hoặc 5 (đếm mọi cặp)
print("đúng:", Solution().mostFrequent(nums, key))  # 100

sai : 5
đúng: 100


### Lỗi 4 — Dùng `count[x] += 1` khi khóa chưa tồn tại

Với `dict` thường, `count[x] += 1` ném `KeyError` ở lần đầu. Hãy dùng `count.get(x, 0) + 1`, `defaultdict(int)` hoặc `Counter`.

In [20]:
count = {}
try:
    count[5] += 1
except KeyError as e:
    print("KeyError:", e)
count[5] = count.get(5, 0) + 1
print(count)

KeyError: 5
{5: 1}


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Cụm từ "**số xuất hiện nhiều nhất**" + "**đứng ngay sau**" ⇒ đếm cặp liền kề bằng hash map.
- Nhìn constraints: `nums[i] <= 1000` ⇒ có thể dùng **mảng đếm** thay dict.
- Muốn gọn: `Counter(b for a, b in zip(nums, nums[1:]) if a == key).most_common(1)[0][0]`.
- Khi duyệt cặp `(i, i+1)`, **luôn** để `range(n - 1)`.
- Bài họ hàng: Top K Frequent Elements, Majority Element, Sort Characters By Frequency.

## 15. Biến thể của bài

1. **Có hòa:** nếu nhiều `target` cùng đếm cao nhất, trả về cái nhỏ nhất → thêm tiêu chí phụ khi chọn.
2. **Đứng ngay trước `key`:** đếm `nums[i-1]` khi `nums[i] == key`.
3. **Top k:** trả về `k` số hay đứng sau `key` nhất → `Counter.most_common(k)`.
4. **Đứng sau trong cửa sổ `w`:** đếm các số trong `nums[i+1 : i+1+w]` khi `nums[i] == key`.
5. **Dạng chuỗi:** đếm ký tự hay đứng sau một ký tự cho trước (bigram đơn giản).

In [21]:
# Biến thể 2: số đứng ngay TRƯỚC key nhiều nhất
def most_frequent_before(nums, key):
    c = Counter(nums[i - 1] for i in range(1, len(nums)) if nums[i] == key)
    return c.most_common(1)[0][0]

print(most_frequent_before([5, 1, 6, 1, 5, 1], 1))   # trước 1: 5,6,5 -> 5

5


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Mỗi `target` duyệt lại mảng | `O(n·u)` | `O(u)` | Hiểu đề, mảng rất nhỏ |
| 2. Dict thủ công | Một lượt, đếm bằng `dict` | `O(n)` | `O(u)` | Nộp bài, rõ ràng, không import |
| 3. Mảng đếm | Dùng mảng 1001 ô theo giá trị | `O(n)` | `O(V)` | Miền giá trị nhỏ, cần tốc độ |
| 4. zip + Counter | Lọc cặp rồi `most_common` | `O(n)` | `O(n)` | Phỏng vấn/viết nhanh, gọn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** Với `nums = [1,2,1,3,1,3]`, `key = 1`, chương trình trả về gì? Hãy nghĩ trước khi chạy ô tiếp theo.

**Bài 2 (tính tay).** Liệt kê các số đứng sau `key` trong `nums = [4,4,5,4,5,4,5]`, `key = 4`, rồi cho biết đáp án.

**Bài 3 (điền TODO).** Hoàn thành hàm `count_followers(nums, key)` trả về `dict` đếm các số đứng sau `key`.

**Bài 4 (tự viết lại).** Tự viết `most_frequent_before(nums, key)` (số đứng ngay trước `key` nhiều nhất) không nhìn code mẫu.

**Bài 5 (mini challenge).** Viết `top_two_followers(nums, key)` trả về danh sách tối đa 2 số hay đứng sau `key` nhất (theo thứ tự giảm dần của số lần).

In [22]:
# Bài 1: chỉ để bạn đoán output rồi chạy kiểm tra
print(Solution().mostFrequent([1, 2, 1, 3, 1, 3], 1))

3


In [23]:
# Bài 3: TODO - KHÔNG được gọi khi run all
def count_followers(nums, key):
    # TODO: trả về dict {số: số lần đứng ngay sau key}
    pass

# Bài 4 & 5: TODO - viết ở đây, chưa gọi
def most_frequent_before_todo(nums, key):
    # TODO
    pass

def top_two_followers(nums, key):
    # TODO
    pass

## 18. Gợi ý 3 mức

**Bài 3**
- Mức 1: Chỉ cần xét `i` từ `0` đến `n - 2`.
- Mức 2: Khi `nums[i] == key`, số cần đếm là `nums[i + 1]`.
- Mức 3: `d[v] = d.get(v, 0) + 1` rồi `return d`.

**Bài 4**
- Mức 1: Đổi vai trò: bây giờ nhìn `nums[i - 1]`.
- Mức 2: `i` chạy từ `1` đến `n - 1` để `i - 1` không âm.
- Mức 3: Đếm `nums[i - 1]` khi `nums[i] == key`, rồi lấy `most_common(1)`.

**Bài 5**
- Mức 1: Dùng `Counter` như cách 4.
- Mức 2: `most_common(2)` trả về danh sách `(giá trị, số lần)`.
- Mức 3: Lấy phần tử đầu mỗi cặp: `[v for v, _ in c.most_common(2)]`.

## 19. Đáp án tham khảo cho bài tập

Mỗi đáp án được kiểm tra bằng `assert`.

In [24]:
# Bài 1: followers của 1 là [2,3,3] -> 3 xuất hiện 2 lần
assert Solution().mostFrequent([1, 2, 1, 3, 1, 3], 1) == 3

# Bài 2: nums = [4,4,5,4,5,4,5], key = 4 -> các số sau key: 4,5,5,5 -> đáp án 5
followers = [[4, 4, 5, 4, 5, 4, 5][i + 1] for i in range(6) if [4, 4, 5, 4, 5, 4, 5][i] == 4]
assert followers == [4, 5, 5, 5], followers
assert Solution().mostFrequent([4, 4, 5, 4, 5, 4, 5], 4) == 5

# Bài 3
def count_followers_ans(nums, key):
    d = {}
    for i in range(len(nums) - 1):
        if nums[i] == key:
            d[nums[i + 1]] = d.get(nums[i + 1], 0) + 1
    return d
assert count_followers_ans([1, 100, 200, 1, 100], 1) == {100: 2}

# Bài 4
def most_frequent_before_ans(nums, key):
    c = Counter(nums[i - 1] for i in range(1, len(nums)) if nums[i] == key)
    return c.most_common(1)[0][0]
assert most_frequent_before_ans([5, 1, 6, 1, 5, 1], 1) == 5

# Bài 5
def top_two_followers_ans(nums, key):
    c = Counter(nums[i + 1] for i in range(len(nums) - 1) if nums[i] == key)
    return [v for v, _ in c.most_common(2)]
assert top_two_followers_ans([1, 7, 1, 7, 1, 8, 1, 9, 1, 9, 1, 9], 1) == [9, 7]
assert top_two_followers_ans([1, 2], 1) == [2]
print("Tất cả đáp án bài tập: OK")

Tất cả đáp án bài tập: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
from typing import List

class Solution:
    def mostFrequent(self, nums: List[int], key: int) -> int:
        count = {}
        for i in range(len(nums) - 1):        # cặp (i, i+1)
            if nums[i] == key:
                v = nums[i + 1]
                count[v] = count.get(v, 0) + 1
        return max(count, key=count.get)
```

### Checklist

- [ ] Vòng lặp chạy `range(len(nums) - 1)`, không vượt biên.
- [ ] Chỉ đếm khi `nums[i] == key`.
- [ ] Đếm số đứng **sau** `key`, không phải đếm chính `key`.
- [ ] Biết `max(d, key=d.get)` hoặc `Counter.most_common(1)`.
- [ ] Đã test `key` ở cuối mảng, `key == target`, mảng độ dài 2.
- [ ] Độ phức tạp `O(n)` thời gian là đủ.